In [27]:
datafold = "NYC" 
stage = "alignment"
sidfile = "Dec-07-2025_vq_semitic_codes"

In [28]:
import pandas as pd
import json
from ast import literal_eval
import os


poi_info = pd.read_csv(f"{datafold}/poi_info.csv")

poi_codes =  pd.read_csv(f"{datafold}/{sidfile}.csv")
# Pid,SID,Vector

# 转换字符串形式的 list
poi_codes["sid"] = poi_codes["sid"].apply(literal_eval)

merged = poi_info.merge(poi_codes, on="pid", how="left")

mapping = {}
for _, row in merged.iterrows():
    code_tuple = tuple(row["sid"])
    code_key = str(list(code_tuple))   # <- JSON 可接受
    mapping[code_key] = {
        "category": row["category"],
        "region": row["region"],
        "latitude": row["latitude"],
        "longitude": row["longitude"],
        "visit_time_and_count": row["visit_time_and_count"]
    }

# 保存映射表
# 检查目录是否存在

if not os.path.exists(f"{datafold}/{stage}/{sidfile}"):
    os.makedirs(f"{datafold}/{stage}/{sidfile}")

with open(f"{datafold}/{stage}/{sidfile}/semantic_code_mapping.json", "w") as f:
    json.dump(mapping, f, indent=4)

print("semantic_code_mapping.json 生成完毕！")

semantic_code_mapping.json 生成完毕！


In [29]:
# 构建对齐数据集

def code_to_tag(code_list):
    letters = "abcdefghijklmnopqrstuvwxyz"
    tags = []
    for i, value in enumerate(code_list):
        tags.append(f"<{letters[i]}_{value}>")
    return "".join(tags)


dataset = []
for code_key, meta in mapping.items():
    code_list = json.loads(code_key.replace("'",'"'))  # 转 list

    tag = code_to_tag(code_list)

    item1 = {
        "instruction": "Given a POI attributes, describe its semantic code.",
        # "instruction": "Can you provide the [SID] based on the attributes?",
        "input": "Can you based on the attributes {" + (
            f"Category: {meta['category']}; "
            f"Region: {meta['region']}; "
            f"Latitude: {meta['latitude']}; "
            f"Longitude: {meta['longitude']}; "
            f"Visit_time_and_count: {meta['visit_time_and_count']}"
        ) + "} to predict the POI semantic code?",
        "output": f"{tag}"
    }
    item2 =  {
        "instruction": "Given a semantic code, describe its POI attributes.",
        "input": f"Can you describe the POI semantic code {tag} attributes?",
        "output": "{" + (
            f"Category: {meta['category']}; "
            f"Region: {meta['region']}; "
            f"Latitude: {meta['latitude']}; "
            f"Longitude: {meta['longitude']}; "
            f"Visit_time_and_count: {meta['visit_time_and_count']}"
        ) + "}"
    }

    dataset.append(item1)
    dataset.append(item2)


# 保存 instruction dataset
with open(f"{datafold}/{stage}/{sidfile}/semantic_instruction_dataset.json", "w") as f:
    json.dump(dataset, f, indent=4)

print("semantic_instruction_dataset.json 生成完毕！")

# 预览前 2 条
print(json.dumps(dataset[:2], indent=4))


semantic_instruction_dataset.json 生成完毕！
[
    {
        "instruction": "Given a POI attributes, describe its semantic code.",
        "input": "Can you based on the attributes {Category: Office; Region: 14; Latitude: 40.7410337; Longitude: -73.99785231; Visit_time_and_count: {15: 3, 17: 2, 18: 2, 16: 1, 19: 1, 21: 1, 20: 1}} to predict the POI semantic code?",
        "output": "<a_5><b_6><c_29><d_0>"
    },
    {
        "instruction": "Given a semantic code, describe its POI attributes.",
        "input": "Can you describe the POI semantic code <a_5><b_6><c_29><d_0> attributes?",
        "output": "{Category: Office; Region: 14; Latitude: 40.7410337; Longitude: -73.99785231; Visit_time_and_count: {15: 3, 17: 2, 18: 2, 16: 1, 19: 1, 21: 1, 20: 1}}"
    }
]


In [30]:
import json
import random  
data =  [] 
datafile =  f"{datafold}/{stage}/{sidfile}/semantic_instruction_dataset.json"
with open(datafile) as f:
    data  = json.load(f)

random.shuffle(data)
train_data =  data[len(data)//10:]
# train_data =  data[:]
valid_data =  data[:len(data)//10]


# 保存文件
train_file = f'{datafold}/{stage}/{sidfile}/train_align.json'
valid_file = f'{datafold}/{stage}/{sidfile}/valid_align.json'

with open(train_file, 'w') as f:
    json.dump(train_data, f, indent=4, ensure_ascii=False)

with open(valid_file, 'w') as f:
    json.dump(valid_data, f, indent=4, ensure_ascii=False)



In [31]:
data =  [] 
with open(train_file) as f:
    data =  json.load(f)
print(len(data))

9243


In [32]:
vdata =  [] 
with open(valid_file) as f:
    vdata =  json.load(f)
print(len(vdata))

1027
